In [2]:
import pandas as pd
import torch
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from huggingface_hub import login
import logging
from transformers import (
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    # tokenizers
    AutoTokenizer,
    DebertaV2Tokenizer,
    DistilBertTokenizer,
    BertTokenizer,
    RobertaTokenizer,
    ElectraTokenizer,
    AlbertTokenizer,
    XLNetTokenizer,
    MobileBertTokenizer,
    # models
    DebertaV2ForSequenceClassification,
    DistilBertForSequenceClassification,
    BertForSequenceClassification,
    RobertaForSequenceClassification,
    ElectraForSequenceClassification,
    AlbertForSequenceClassification,
    XLNetForSequenceClassification,
    MobileBertForSequenceClassification,
)
from torch.nn import CrossEntropyLoss
# evaluation metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from collections import Counter

import transformers
print(transformers.__version__)
print(transformers.TrainingArguments)

# Cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

4.29.2
<class 'transformers.training_args.TrainingArguments'>


In [ ]:
import pandas as pd
import numpy as np
import os
import json

from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer
from statsmodels.stats.contingency_tables import mcnemar  # pip install statsmodels if needed

# --- Load performance CSV (same as relavance_classifier_multi_nvidia.ipynb) ---
if os.path.exists("model_performance.csv"):
    perf_df = pd.read_csv("model_performance.csv")
elif os.path.exists("model_performance_nvidia.csv"):
    perf_df = pd.read_csv("model_performance_nvidia.csv")
else:
    raise FileNotFoundError(
        "Expected model_performance.csv or model_performance_nvidia.csv in the working directory."
    )

# Make sure model name is string
perf_df["model"] = perf_df["model"].astype(str)

# For each model, pick the row with the BEST F1
best_rows = (
    perf_df.sort_values(["model", "f1"], ascending=[True, False])
           .groupby("model", as_index=False)
           .first()
)

print("Best configuration per model (by F1):")
display(best_rows)

# List of (model_name, max_length) to use for statistical comparison
MODEL_RUNS = [(row["model"], int(row["max_length"])) for _, row in best_rows.iterrows()]
print("\nModel runs to evaluate:", MODEL_RUNS)

# Build (model_name, max_length) -> best_checkpoint path for loading exact checkpoints in statistical tests
best_checkpoint_map = {}
if "best_checkpoint" in perf_df.columns:
    for _, row in perf_df.iterrows():
        ckpt = row.get("best_checkpoint")
        if pd.notna(ckpt) and ckpt and str(ckpt).strip():
            best_checkpoint_map[(str(row["model"]), int(row["max_length"]))] = str(ckpt).strip()
else:
    for alt_name in ("model_performance.csv", "model_performance_nvidia.csv"):
        if not os.path.exists(alt_name):
            continue
        try:
            alt = pd.read_csv(alt_name)
            if "best_checkpoint" not in alt.columns:
                continue
            alt["model"] = alt["model"].astype(str)
            for _, row in alt.iterrows():
                ckpt = row.get("best_checkpoint")
                if pd.notna(ckpt) and ckpt and str(ckpt).strip():
                    best_checkpoint_map[(str(row["model"]), int(row["max_length"]))] = str(ckpt).strip()
            break
        except Exception:
            pass
print(f"Checkpoint map has {len(best_checkpoint_map)} (model, max_length) entries from CSV.")


Best configuration per model (by F1):


,model,max_length,train_time_sec,pred_time_sec,accuracy,f1,precision,recall
0,albert-base-v1,512,434.705156,9.194373,0.979790,0.979653,0.979585,0.979790
1,albert-base-v2,512,535.062250,12.144323,0.978981,0.978810,0.978738,0.978981
2,albert-large-v2,128,316.294875,6.968168,0.966047,0.965674,0.965473,0.966047
3,albert-xlarge-v2,128,1801.389500,23.367988,0.870655,0.810454,0.758040,0.870655
4,bert-large-uncased,512,815.210000,29.074219,0.976556,0.976061,0.976215,0.976556
5,deberta-large,128,2064.516125,46.919852,0.983832,0.983700,0.983670,0.983832
6,deberta-v3-base,512,679.918250,17.636084,0.981407,0.981123,0.981187,0.981407
7,deberta-v3-large,512,2057.726250,45.725594,0.979790,0.979363,0.979594,0.979790
8,deberta-v3-small,128,106.569336,2.781707,0.982215,0.981866,0.982092,0.982215
9,deberta-xlarge,128,4090.735250,102.445656,0.978173,0.977580,0.978060,0.978173



Model runs to evaluate: [('albert-base-v1', 512), ('albert-base-v2', 512), ('albert-large-v2', 128), ('albert-xlarge-v2', 128), ('bert-large-uncased', 512), ('deberta-large', 128), ('deberta-v3-base', 512), ('deberta-v3-large', 512), ('deberta-v3-small', 128), ('deberta-xlarge', 128), ('distilbert', 256), ('electra-base-generator', 512), ('electra-large', 128), ('electra-small', 128), ('mobilebert', 512), ('roberta', 128), ('roberta-large', 256), ('xlm-roberta-base', 512), ('xlm-roberta-large', 128), ('xlnet', 256)]


In [3]:
# ---- Tuning parameters ----
# Kept in sync with relavance_classifier_multi_nvidia.ipynb (only output_dir is used here for checkpoint fallback).
# Size tiers and LRs follow convention from: Devlin et al. 2019 (BERT), Mosbach et al. 2021 (fine-tuning stability).
# Small <100M, Base/Medium ~110M, Large 300M+. Larger models use slightly lower LR for stable convergence.
CONFIG = {
    # epochs tuned by model size/type
    "epochs": 3,
    # batch sizes tuned by model size/type
    "batch_size": {
        "small": 16,
        "medium": 8,
        "large": 4
    },
    "max_length": [128, 256, 512],  # Max length of input sequences
    # learning rates by size (small -> higher, large -> lower; Devlin et al. 2e-5--5e-5)
    "learning_rate": {
        "small": 5e-5,
        "medium": 3e-5,
        "large": 2e-5
    },
    "weight_decay": 0.01,  # Weight decay for regularization
    "output_dir": "D:/huggingface_cache/classification_models"
}

# ---- Model configurations ----
# Ordered roughly by parameter count (small -> xxl). Each model has a 'type' key.
MODEL_CONFIGS = {
    # ---- XS (very small, <= ~20M) ----
    "albert-base-v2": {
        "type": "small",
        "tokenizer_class": AlbertTokenizer,
        "pretrained_model_name": "albert-base-v2",  # ~11M
        "model_class": AlbertForSequenceClassification
    },
    "electra-small": {
        "type": "small",
        "tokenizer_class": ElectraTokenizer,
        "pretrained_model_name": "google/electra-small-discriminator",  # ~14M
        "model_class": ElectraForSequenceClassification
    },
    # ---- S (small, ~20M-70M) ----
    "mobilebert": {
        "type": "small",
        "tokenizer_class": AutoTokenizer,
        "pretrained_model_name": "google/mobilebert-uncased",  # ~25M
        "model_class": MobileBertForSequenceClassification
    },
    "deberta-v3-small": {
        "type": "small",
        "tokenizer_class": DebertaV2Tokenizer,
        "pretrained_model_name": "microsoft/deberta-v3-small",  # ~55M
        "model_class": DebertaV2ForSequenceClassification
    },
    "distilbert": {
        "type": "small",
        "tokenizer_class": DistilBertTokenizer,
        "pretrained_model_name": "distilbert-base-uncased",  # ~66M
        "model_class": DistilBertForSequenceClassification
    },

    # ---- M (medium, ~70M-200M) ----
    "bert": {
        "type": "medium",
        "tokenizer_class": BertTokenizer,
        "pretrained_model_name": "bert-base-uncased",  # ~110M
        "model_class": BertForSequenceClassification
    },
    "xlnet": {
        "type": "medium",
        "tokenizer_class": XLNetTokenizer,
        "pretrained_model_name": "xlnet-base-cased",  # ~110M
        "model_class": XLNetForSequenceClassification
    },
    "roberta": {
        "type": "medium",
        "tokenizer_class": RobertaTokenizer,
        "pretrained_model_name": "roberta-base",  # ~125M
        "model_class": RobertaForSequenceClassification
    },
    "deberta-v3-base": {
        "type": "medium",
        "tokenizer_class": AutoTokenizer,
        "pretrained_model_name": "microsoft/deberta-v3-base",  # ~140M
        "model_class": AutoModelForSequenceClassification
    },
    "xlm-roberta-base": {
        "type": "medium",
        "tokenizer_class": AutoTokenizer,
        "pretrained_model_name": "xlm-roberta-base",  # ~270M
        "model_class": AutoModelForSequenceClassification
    },
    "bert-large-uncased": {
        "type": "large",
        "tokenizer_class": BertTokenizer,
        "pretrained_model_name": "bert-large-uncased",  # ~340M
        "model_class": BertForSequenceClassification
    },
    "roberta-large": {
        "type": "large",
        "tokenizer_class": RobertaTokenizer,
        "pretrained_model_name": "roberta-large",  # ~355M
        "model_class": RobertaForSequenceClassification
    },
    "electra-large": {
        "type": "large",
        "tokenizer_class": ElectraTokenizer,
        "pretrained_model_name": "google/electra-large-discriminator",  # ~335M
        "model_class": ElectraForSequenceClassification
    },
    # ---- XL (extra large, ~400M-800M) ----
    "xlm-roberta-large": {
        "type": "large",
        "tokenizer_class": AutoTokenizer,
        "pretrained_model_name": "xlm-roberta-large",  # ~550M
        "model_class": AutoModelForSequenceClassification
    },
    # ---- XXL (very large, > ~800M) ----
    "deberta-large": {
        "type": "large",
        "tokenizer_class": AutoTokenizer,
        "pretrained_model_name": "microsoft/deberta-large",  # very large (~>800M depending on variant)
        "model_class": AutoModelForSequenceClassification
    }
}


In [8]:
import re


def get_best_checkpoint(model_name):
    """
    Fix: scans all checkpoint-* folders under the model directory
    and reads trainer_state.json to find the best checkpoint.
    """
    model_dir = f"{CONFIG['output_dir']}/{model_name}"

    # Find all folders named checkpoint-xxxx
    checkpoint_dirs = [
        os.path.join(model_dir, d)
        for d in os.listdir(model_dir)
        if os.path.isdir(os.path.join(model_dir, d)) and re.match(r"checkpoint-\d+", d)
    ]

    if len(checkpoint_dirs) == 0:
        raise FileNotFoundError(f"No checkpoint-* folders found for {model_name} in {model_dir}")

    best_ckpt = None
    best_metric = -9999

    for ckpt_dir in checkpoint_dirs:
        state_file = os.path.join(ckpt_dir, "trainer_state.json")
        if not os.path.exists(state_file):
            continue

        with open(state_file, "r") as f:
            state = json.load(f)

        # Try to get best_metric; fallback to "eval_f1" if present
        metric = state.get("best_metric", None)
        if metric is None:
            # look in logs
            logs = state.get("log_history", [])
            for log in logs:
                if "eval_f1" in log:
                    metric = log["eval_f1"]

        if metric is None:
            continue

        if metric > best_metric:
            best_metric = metric
            best_ckpt = ckpt_dir

    if best_ckpt is None:
        raise ValueError(f"No valid checkpoint with metrics found for {model_name}")

    print(f"[BEST CHECKPOINT] {model_name} → {best_ckpt} (best_metric={best_metric})")
    return best_ckpt


def get_checkpoint_for_run(model_name, max_len):
    """
    Return the checkpoint path for (model_name, max_length).
    Uses best_checkpoint_map from CSV when available; otherwise falls back to get_best_checkpoint(model_name).
    """
    key = (model_name, max_len)
    if key in best_checkpoint_map:
        path = best_checkpoint_map[key]
        path = os.path.normpath(path) if isinstance(path, str) else path
        print(f"[BEST CHECKPOINT] {model_name} (max_len={max_len}) → {path} (from CSV)")
        return path
    return get_best_checkpoint(model_name)

In [5]:
# Load the labeled chunks
with open("exported_chunks.jsonl", "r", encoding="utf-8") as f:
    labeled_chunks = [json.loads(line) for line in f]

data = pd.DataFrame(labeled_chunks)
labeled_count = data['label'].value_counts().to_dict()

# Get the first 9000 rows
data = data.head(9000)

# Remove rows with label == 11
data = data[data['label'] != 11]

# Print labeled count after removing label 11
labeled_count = data['label'].value_counts().to_dict()
print(f"Labeled chunks after removing label 11: {labeled_count}")

# Remove rows where label == 1 and text length < 100
data = data[~((data['label'] == 1) & (data['text'].str.len() < 100))]

# Print final labeled count
labeled_count = data['label'].value_counts().to_dict()
print(f"Final labeled chunks: {labeled_count}")


Labeled chunks after removing label 11: {1: 8199, 0: 800}
Final labeled chunks: {1: 5384, 0: 800}


In [6]:
# Loading the data
data['label'] = data['label'].astype(int)

# Train-Test Split using stratified sampling
train_df, test_df = train_test_split(data, test_size=0.2, stratify=data['label'], random_state=42)

# since there is a class imbalance, we will compute class weights
# to handle this in the loss function
labels = train_df["label"].values
# Compute class weights
classes = np.unique(labels)
weights = compute_class_weight(class_weight="balanced",
                            classes=classes,
                            y=labels)
class_weights = torch.tensor(weights, dtype=torch.float, device=device)
print("Class weights:", class_weights)

# Convert ing the DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

Class weights: tensor([3.8648, 0.5743], device='cuda:0')


In [10]:
all_predictions = {}
y_true_global = None  # we'll store y_true once
import re
def regenerate_predictions(model_name, max_len):
    global y_true_global

    print(f"\n► Regenerating predictions for: {model_name} (max_len={max_len})")

    cfg = MODEL_CONFIGS[model_name]
    best_ckpt = get_checkpoint_for_run(model_name, max_len)

    # Load tokenizer & model from best checkpoint
    tokenizer = cfg["tokenizer_class"].from_pretrained(cfg["pretrained_model_name"])
    model = cfg["model_class"].from_pretrained(
        best_ckpt,
        num_labels=len(data['label'].unique())
    ).to(device)

    # Tokenize test set
    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_len
        )

    test_enc = test_dataset.map(tokenize_fn, batched=True)
    test_enc = test_enc.rename_column("label", "labels")
    test_enc.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    # Use Trainer for prediction only
    trainer = Trainer(model=model, tokenizer=tokenizer)
    preds = trainer.predict(test_enc)

    y_true = preds.label_ids
    y_pred = preds.predictions.argmax(-1)
    y_prob = preds.predictions  # logits

    if y_true_global is None:
        y_true_global = y_true
        np.save("y_true.npy", y_true)
    else:
        # sanity check
        assert np.array_equal(y_true_global, y_true), "y_true changed across models!"

    key = f"{model_name}_len{max_len}"
    all_predictions[key] = {
        "model": model_name,
        "max_length": max_len,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    # Save to disk
    np.save(f"preds_{key}.npy", y_pred)
    np.save(f"probs_{key}.npy", y_prob)

    return key


# ---- Run for selected (model, max_length) combos ----
model_keys = []
for model_name, max_len in MODEL_RUNS:
    try:
        key = regenerate_predictions(model_name, max_len)
        model_keys.append(key)
    except Exception as e:
        print(f"⚠️ Skipping {model_name} (len={max_len}) due to error: {e}")

print("\nFinished regenerating predictions for keys:")
print(model_keys)



► Regenerating predictions for: albert-base-v1 (max_len=512)
[BEST CHECKPOINT] albert-base-v1 → huggingface_cache/classification_models/albert-base-v1\checkpoint-620 (best_metric=0.9796528180185096)


c:\Users\ctngweru\AppData\Local\anaconda3\envs\llm-forge-Copy\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

c:\Users\ctngweru\AppData\Local\anaconda3\envs\llm-forge-Copy\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ctngweru\.cache\huggingface\hub\models--albert-base-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

c:\Users\ctngweru\AppData\Local\anaconda3\envs\llm-forge-Copy\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ctngweru\.cache\huggingface\hub\models--xlnet-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

Map:   0%|          | 0/1237 [00:00<?, ? examples/s]

  0%|          | 0/155 [00:00<?, ?it/s]


Finished regenerating predictions for keys:
['albert-base-v1_len512', 'albert-base-v2_len512', 'albert-large-v2_len128', 'albert-xlarge-v2_len128', 'bert-large-uncased_len512', 'deberta-large_len128', 'deberta-v3-base_len512', 'deberta-v3-large_len512', 'deberta-v3-small_len128', 'deberta-xlarge_len128', 'distilbert_len256', 'electra-base-generator_len512', 'electra-large_len128', 'electra-small_len128', 'mobilebert_len512', 'roberta_len128', 'roberta-large_len256', 'xlm-roberta-base_len512', 'xlm-roberta-large_len128', 'xlnet_len256']


In [12]:
from sklearn.metrics import precision_recall_fscore_support

def metric_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def metric_f1_weighted(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted")

def bootstrap_ci_for_model(key, metric_fn, n_boot=2000, random_state=42):
    rng = np.random.default_rng(random_state)
    y_true = all_predictions[key]["y_true"]
    y_pred = all_predictions[key]["y_pred"]
    N = len(y_true)

    stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, N, N)
        stats[i] = metric_fn(y_true[idx], y_pred[idx])

    point_est = metric_fn(y_true, y_pred)
    low = np.percentile(stats, 2.5)
    high = np.percentile(stats, 97.5)
    return point_est, low, high

# ---- Build summary table with bootstrap CIs ----
bootstrap_rows = []
for key in model_keys:
    mname = all_predictions[key]["model"]
    mlen = all_predictions[key]["max_length"]

    acc, acc_low, acc_high = bootstrap_ci_for_model(key, metric_accuracy)
    f1, f1_low, f1_high = bootstrap_ci_for_model(key, metric_f1_weighted)

    bootstrap_rows.append({
        "key": key,
        "model": mname,
        "max_length": mlen,
        "accuracy": acc,
        "accuracy_ci_low": acc_low,
        "accuracy_ci_high": acc_high,
        "f1_weighted": f1,
        "f1_ci_low": f1_low,
        "f1_ci_high": f1_high,
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)
print("\nBootstrap CI summary:")
display(bootstrap_df)

bootstrap_df.to_csv("model_bootstrap_ci_results.csv", index=False)



Bootstrap CI summary:


,key,model,max_length,accuracy,accuracy_ci_low,accuracy_ci_high,f1_weighted,f1_ci_low,f1_ci_high
0,albert-base-v1_len512,albert-base-v1,512,0.979790,0.971686,0.987065,0.979653,0.970968,0.987030
1,albert-base-v2_len512,albert-base-v2,512,0.978981,0.970897,0.986277,0.978810,0.970498,0.986406
2,albert-large-v2_len128,albert-large-v2,128,0.954729,0.942603,0.965259,0.951228,0.937456,0.963578
3,albert-xlarge-v2_len128,albert-xlarge-v2,128,0.870655,0.851253,0.889248,0.810454,0.782855,0.837119
4,bert-large-uncased_len512,bert-large-uncased,512,0.976556,0.967664,0.983852,0.976061,0.966771,0.983751
5,deberta-large_len128,deberta-large,128,0.982215,0.974131,0.989491,0.981866,0.973345,0.989248
6,deberta-v3-base_len512,deberta-v3-base,512,0.981407,0.973323,0.988682,0.981123,0.972613,0.988524
7,deberta-v3-large_len512,deberta-v3-large,512,0.979790,0.971706,0.987065,0.979363,0.970804,0.986899
8,deberta-v3-small_len128,deberta-v3-small,128,0.978981,0.970089,0.986257,0.978751,0.969914,0.986127
9,deberta-xlarge_len128,deberta-xlarge,128,0.976556,0.967664,0.984640,0.976061,0.966431,0.984352


In [11]:
import itertools

def mcnemar_two(key_a, key_b):
    y_true = all_predictions[key_a]["y_true"]
    A = all_predictions[key_a]["y_pred"]
    B = all_predictions[key_b]["y_pred"]

    # b = A wrong, B correct
    # c = A correct, B wrong
    b_err = np.sum((A != y_true) & (B == y_true))
    c_err = np.sum((A == y_true) & (B != y_true))

    table = [[0, b_err],
             [c_err, 0]]

    result = mcnemar(table, exact=False, correction=True)
    return result.statistic, result.pvalue, b_err, c_err

pair_results = []
for key_a, key_b in itertools.combinations(model_keys, 2):
    stat, pval, b_err, c_err = mcnemar_two(key_a, key_b)

    pair_results.append({
        "model_a": all_predictions[key_a]["model"],
        "max_len_a": all_predictions[key_a]["max_length"],
        "key_a": key_a,
        "model_b": all_predictions[key_b]["model"],
        "max_len_b": all_predictions[key_b]["max_length"],
        "key_b": key_b,
        "mcnemar_chi2": stat,
        "p_value": pval,
        "b_err_A_wrong_B_right": b_err,
        "c_err_A_right_B_wrong": c_err,
    })

mcnemar_df = pd.DataFrame(pair_results)
print("\nPairwise McNemar test results:")
display(mcnemar_df.head())

mcnemar_df.to_csv("mcnemar_pairwise_results.csv", index=False)



Pairwise McNemar test results:


c:\Users\ctngweru\AppData\Local\anaconda3\envs\llm-forge-Copy\lib\site-packages\statsmodels\stats\contingency_tables.py:1348: RuntimeWarning: divide by zero encountered in scalar divide
  statistic = (np.abs(n1 - n2) - corr)**2 / (1. * (n1 + n2))


,model_a,max_len_a,key_a,model_b,max_len_b,key_b,mcnemar_chi2,p_value,b_err_A_wrong_B_right,c_err_A_right_B_wrong
0,albert-base-v1,512,albert-base-v1_len512,albert-base-v2,512,albert-base-v2_len512,0.000000,1.000000e+00,8,9
1,albert-base-v1,512,albert-base-v1_len512,albert-large-v2,128,albert-large-v2_len128,18.367347,1.821530e-05,9,40
2,albert-base-v1,512,albert-base-v1_len512,albert-xlarge-v2,128,albert-xlarge-v2_len128,115.845161,5.138930e-27,10,145
3,albert-base-v1,512,albert-base-v1_len512,bert-large-uncased,512,bert-large-uncased_len512,0.562500,4.532547e-01,6,10
4,albert-base-v1,512,albert-base-v1_len512,deberta-large,128,deberta-large_len128,0.266667,6.055766e-01,9,6
